<a href="https://colab.research.google.com/github/roshiend/Poser-Outfit-Changer/blob/main/Pose_Cloth_Changer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Pose & Cloth Swap (Free Google Colab)

Transfer **pose + outfit** from a reference image onto a **base person**, while keeping face identity and body proportions as consistent as possible.

**Pipeline:** Leffa VTON (outfit) → Leffa pose transfer → InsightFace face lock  
**Cost:** $0 — free Colab T4 GPU + open-source models

### Before you run
1. **Runtime → Change runtime type → T4 GPU**
2. Run cells **in order** (install runs *before* importing packages, so you should **not** need Restart session)
3. First run downloads multi-GB weights
4. Open the Gradio public link when it appears


## 1. Install dependencies (do this first)

Installs only what’s missing. **Does not reinstall NumPy/Pandas** when Colab’s versions are already fine — that avoids the “restart session” popup.

If you still see a restart warning after an old broken run: **Runtime → Disconnect and delete runtime**, reopen, then run from here.


In [ ]:
import os
import sys
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

WORK = Path("/content/pose_cloth_swap")
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
print("Working dir:", WORK)

# Shell only - avoid importing numpy/torch before installs settle
!nvidia-smi

def pkg_ver(name: str):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

def parse_ver(v: str):
    parts = []
    for p in v.split(".")[:3]:
        digits = "".join(ch for ch in p if ch.isdigit())
        parts.append(int(digits) if digits else 0)
    while len(parts) < 3:
        parts.append(0)
    return tuple(parts)

# Keep Colab's numpy/pandas when already compatible (prevents restart-session warning)
numpy_v = pkg_ver("numpy")
pandas_v = pkg_ver("pandas")
print("preinstalled numpy:", numpy_v, "| pandas:", pandas_v)

pins = []
if numpy_v is None or not (parse_ver("1.26.0") <= parse_ver(numpy_v) < parse_ver("2.1.0")):
    pins.append('"numpy>=1.26.4,<2.1"')
else:
    print("Keeping existing numpy", numpy_v)

if pandas_v is None or not (parse_ver("2.0.0") <= parse_ver(pandas_v) < parse_ver("2.3.0")):
    pins.append('"pandas>=2.0,<2.3"')
else:
    print("Keeping existing pandas", pandas_v)

# App deps — no --upgrade/--force-reinstall
pkgs = " ".join(pins + [
    '"gradio>=4.44,<6"',
    '"huggingface_hub>=0.24,<1"',
    "accelerate", "cloudpickle", "diffusers", "einops", "fvcore",
    "imageio", "matplotlib", "omegaconf", "opencv-python-headless", "peft", "pycocotools",
    "regex", "safetensors", "scikit-image", "scipy", "timm", "transformers",
    "insightface", "onnx", "onnxruntime",
    "pyyaml", "iopath", "cloudpathlib", "portalocker", "ninja",
])
print("Installing:", pkgs)
get_ipython().run_line_magic("pip", f"install -q {pkgs}")

# Detectron2: prefer fast Colab path (clone + PYTHONPATH), build only if needed
def detectron2_ok():
    try:
        import detectron2
        from detectron2 import _C  # noqa: F401
        print("detectron2 OK:", detectron2.__file__)
        return True
    except Exception as e:
        print("detectron2 not ready:", repr(e))
        return False

if not detectron2_ok():
    print("Setting up detectron2 (clone + path, usually faster than a full build)...")
    d2 = WORK / "detectron2_repo"
    if not d2.exists():
        !git clone --depth 1 https://github.com/facebookresearch/detectron2.git {d2}
    d2_path = str(d2.resolve())
    if d2_path in sys.path:
        sys.path.remove(d2_path)
    sys.path.insert(0, d2_path)
    (WORK / "detectron2_path.txt").write_text(d2_path)
    get_ipython().run_line_magic("pip", "install -q ninja iopath fvcore cloudpathlib portalocker omegaconf")

if not detectron2_ok():
    print("Building detectron2 from source (slower)...")
    get_ipython().run_line_magic(
        "pip",
        'install -q --no-build-isolation "git+https://github.com/facebookresearch/detectron2.git"',
    )

if not detectron2_ok():
    raise RuntimeError(
        "detectron2 install failed. Runtime → Disconnect and delete runtime, "
        "reopen with T4 GPU, run this cell again."
    )

print("Deps installed — continue to section 2.")
print("If Colab still shows a restart banner from an OLD cell, ignore it only if you did NOT reinstall numpy above.")


## 2. Check GPU


In [ ]:
import torch
print("torch:", torch.__version__, "| cuda:", torch.version.cuda)
assert torch.cuda.is_available(), "Enable a GPU: Runtime → Change runtime type → T4 GPU"
print("CUDA OK:", torch.cuda.get_device_name(0), "| VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1), "GB")

import numpy as np
print("numpy:", np.__version__)


## 3. Clone Leffa + write helpers


In [ ]:
import os
from pathlib import Path

WORK = Path("/content/pose_cloth_swap")
os.chdir(WORK)

if not (WORK / "Leffa").exists():
    !git clone --depth 1 https://github.com/franciszzj/Leffa.git
else:
    print("Leffa already cloned")

leffa = WORK / "Leffa"

# Keep SCHP + densepose from Leffa 3rdparty. Prefer pip detectron2 over vendored copy.
for name in ("SCHP", "densepose"):
    link = leffa / name
    target = leffa / "3rdparty" / name
    if not target.exists():
        print(f"WARNING: missing {target}")
        continue
    if link.is_dir() and any(link.iterdir()):
        print(f"OK: {name}")
        continue
    if link.exists() or link.is_symlink():
        if link.is_symlink() or link.is_file():
            link.unlink()
        elif link.is_dir():
            link.rmdir()
    try:
        link.symlink_to(target, target_is_directory=True)
        print(f"Linked {name} -> 3rdparty/{name}")
    except OSError:
        import shutil
        shutil.copytree(target, link)
        print(f"Copied 3rdparty/{name} -> {name}")

d2 = leffa / "detectron2"
if d2.is_symlink() or (d2.exists() and d2.is_file()):
    d2.unlink()
    print("Removed Leffa/detectron2 symlink (using pip detectron2)")
elif d2.is_dir() and not any(d2.iterdir()):
    d2.rmdir()
    print("Removed empty Leffa/detectron2 dir")

print("Leffa ready at", leffa)


In [ ]:
from pathlib import Path
WORK = Path("/content/pose_cloth_swap")
Path(WORK / "pipeline").mkdir(parents=True, exist_ok=True)
print("pipeline/ ready")


In [ ]:
%%writefile /content/pose_cloth_swap/pipeline/memory.py
"""VRAM helpers for sequential model loading on Colab T4."""

from __future__ import annotations

import gc


def free_vram() -> None:
    """Release unused CUDA memory as aggressively as possible."""
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass


In [ ]:
%%writefile /content/pose_cloth_swap/pipeline/face_lock.py
"""Paste the base person's face onto a generated image to preserve identity."""

from __future__ import annotations

from typing import Tuple

import cv2
import numpy as np
from PIL import Image


def _to_bgr(image: Image.Image | np.ndarray) -> np.ndarray:
    if isinstance(image, Image.Image):
        rgb = np.array(image.convert("RGB"))
    else:
        rgb = image
        if rgb.ndim == 2:
            rgb = cv2.cvtColor(rgb, cv2.COLOR_GRAY2RGB)
        elif rgb.shape[2] == 4:
            rgb = cv2.cvtColor(rgb, cv2.COLOR_RGBA2RGB)
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)


def _to_pil(bgr: np.ndarray) -> Image.Image:
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return Image.fromarray(rgb)


def _largest_face(faces):
    if not faces:
        return None
    return max(
        faces,
        key=lambda f: float((f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1])),
    )


def _face_ellipse_mask(
    shape: Tuple[int, int],
    bbox: np.ndarray,
    feather: float = 0.35,
) -> np.ndarray:
    h, w = shape[:2]
    x1, y1, x2, y2 = [int(v) for v in bbox]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w - 1, x2), min(h - 1, y2)
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    # Slightly enlarge vertically to cover forehead / chin.
    ax = max(1.0, (x2 - x1) * 0.52)
    ay = max(1.0, (y2 - y1) * 0.62)
    mask = np.zeros((h, w), dtype=np.float32)
    cv2.ellipse(
        mask,
        (int(cx), int(cy)),
        (int(ax), int(ay)),
        0,
        0,
        360,
        1.0,
        -1,
    )
    k = max(3, int(min(ax, ay) * feather) | 1)
    mask = cv2.GaussianBlur(mask, (k, k), 0)
    return mask


def lock_face_identity(
    base_image: Image.Image | np.ndarray,
    generated_image: Image.Image | np.ndarray,
    face_app=None,
    blend: float = 0.92,
) -> Image.Image:
    """
    Overlay the base face onto the generated image with a feathered blend.

    Falls back to returning the generated image unchanged if faces cannot be
    detected or InsightFace is unavailable.
    """
    base_bgr = _to_bgr(base_image)
    gen_bgr = _to_bgr(generated_image)

    if face_app is None:
        try:
            from insightface.app import FaceAnalysis

            face_app = FaceAnalysis(
                name="buffalo_l",
                providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
            )
            face_app.prepare(ctx_id=0, det_size=(640, 640))
        except Exception as exc:
            print(f"[face_lock] InsightFace unavailable ({exc}); skipping face lock.")
            return (
                generated_image
                if isinstance(generated_image, Image.Image)
                else Image.fromarray(
                    cv2.cvtColor(_to_bgr(generated_image), cv2.COLOR_BGR2RGB)
                )
            )

    base_faces = face_app.get(base_bgr)
    gen_faces = face_app.get(gen_bgr)
    src_face = _largest_face(base_faces)
    dst_face = _largest_face(gen_faces)

    if src_face is None or dst_face is None:
        print("[face_lock] Face not found on base or result; skipping face lock.")
        return _to_pil(gen_bgr)

    src_pts = np.float32(src_face.kps)
    dst_pts = np.float32(dst_face.kps)
    matrix, _ = cv2.estimateAffinePartial2D(src_pts, dst_pts, method=cv2.LMEDS)
    if matrix is None:
        print("[face_lock] Could not estimate face warp; skipping.")
        return _to_pil(gen_bgr)

    warped = cv2.warpAffine(
        base_bgr,
        matrix,
        (gen_bgr.shape[1], gen_bgr.shape[0]),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT_101,
    )

    # Warp base bbox into generated space for the blend mask.
    x1, y1, x2, y2 = src_face.bbox
    corners = np.float32([[x1, y1], [x2, y1], [x2, y2], [x1, y2]]).reshape(-1, 1, 2)
    warped_corners = cv2.transform(corners, matrix).reshape(-1, 2)
    wx1, wy1 = warped_corners.min(axis=0)
    wx2, wy2 = warped_corners.max(axis=0)
    mask = _face_ellipse_mask(gen_bgr.shape, np.array([wx1, wy1, wx2, wy2]))
    mask = np.clip(mask * float(blend), 0.0, 1.0)[..., None]

    # Match color loosely inside the face region.
    m = mask[..., 0] > 0.05
    if np.any(m):
        for c in range(3):
            src_mean = float(warped[..., c][m].mean())
            dst_mean = float(gen_bgr[..., c][m].mean())
            if src_mean > 1e-3:
                warped[..., c] = np.clip(
                    warped[..., c].astype(np.float32) * (dst_mean / src_mean),
                    0,
                    255,
                ).astype(np.uint8)

    out = (
        warped.astype(np.float32) * mask + gen_bgr.astype(np.float32) * (1.0 - mask)
    ).astype(np.uint8)
    return _to_pil(out)


In [ ]:
%%writefile /content/pose_cloth_swap/pipeline/leffa_sequential.py
"""
Sequential Leffa runner for Colab T4 (~15GB).

Loads at most one diffusion checkpoint at a time:
  VTON -> unload -> Pose transfer -> unload -> face lock.
"""

from __future__ import annotations

import sys
from pathlib import Path
from typing import Literal, Optional, Tuple, Union

import numpy as np
from PIL import Image

from .face_lock import lock_face_identity
from .memory import free_vram

PathLike = Union[str, Path, Image.Image]
Mode = Literal["both", "outfit_only", "pose_only"]


def _ensure_leffa_on_path(leffa_root: Path) -> None:
    root = str(leffa_root.resolve())
    if root not in sys.path:
        sys.path.insert(0, root)


def _as_pil(image: PathLike) -> Image.Image:
    if isinstance(image, Image.Image):
        return image.convert("RGB")
    return Image.open(image).convert("RGB")


class PoseClothPipeline:
    """
    Identity-safe pose + outfit transfer.

    Inputs
    ------
    base_image : person whose face / body proportions must stay consistent
    ref_image  : source of outfit and/or pose
    """

    def __init__(
        self,
        leffa_root: str | Path = "./Leffa",
        ckpt_dir: str | Path | None = None,
        dtype: str = "float16",
        enable_face_lock: bool = True,
    ) -> None:
        self.leffa_root = Path(leffa_root)
        self.ckpt_dir = Path(ckpt_dir) if ckpt_dir else self.leffa_root / "ckpts"
        self.dtype = dtype
        self.enable_face_lock = enable_face_lock

        _ensure_leffa_on_path(self.leffa_root)

        # Lazy-loaded preprocess / model handles
        self._parsing = None
        self._openpose = None
        self._densepose = None
        self._vt_inference = None
        self._pt_inference = None
        self._face_app = None
        self._active_model: Optional[str] = None

    # ------------------------------------------------------------------ setup
    def download_checkpoints(self) -> None:
        from huggingface_hub import snapshot_download

        self.ckpt_dir.mkdir(parents=True, exist_ok=True)
        print("Downloading Leffa checkpoints (first run can take a while)...")
        snapshot_download(repo_id="franciszzj/Leffa", local_dir=str(self.ckpt_dir))
        print("Checkpoints ready at", self.ckpt_dir)

    def _load_preprocessors(self) -> None:
        if self._parsing is not None:
            return

        from leffa_utils.densepose_predictor import DensePosePredictor
        from preprocess.humanparsing.run_parsing import Parsing
        from preprocess.openpose.run_openpose import OpenPose

        ckpt = self.ckpt_dir
        self._parsing = Parsing(
            atr_path=str(ckpt / "humanparsing" / "parsing_atr.onnx"),
            lip_path=str(ckpt / "humanparsing" / "parsing_lip.onnx"),
        )
        self._openpose = OpenPose(
            body_model_path=str(ckpt / "openpose" / "body_pose_model.pth"),
        )
        self._densepose = DensePosePredictor(
            config_path=str(ckpt / "densepose" / "densepose_rcnn_R_50_FPN_s1x.yaml"),
            weights_path=str(ckpt / "densepose" / "model_final_162be9.pkl"),
        )

    def _unload_diffusion(self) -> None:
        self._vt_inference = None
        self._pt_inference = None
        self._active_model = None
        free_vram()

    def _load_vton(self, model_type: str = "viton_hd") -> None:
        if self._active_model == f"vton_{model_type}" and self._vt_inference is not None:
            return
        self._unload_diffusion()

        from leffa.inference import LeffaInference
        from leffa.model import LeffaModel

        weight = (
            self.ckpt_dir / "virtual_tryon.pth"
            if model_type == "viton_hd"
            else self.ckpt_dir / "virtual_tryon_dc.pth"
        )
        model = LeffaModel(
            pretrained_model_name_or_path=str(
                self.ckpt_dir / "stable-diffusion-inpainting"
            ),
            pretrained_model=str(weight),
            dtype=self.dtype,
        )
        self._vt_inference = LeffaInference(model=model)
        self._active_model = f"vton_{model_type}"
        print(f"Loaded VTON model: {model_type}")

    def _load_pose(self) -> None:
        if self._active_model == "pose" and self._pt_inference is not None:
            return
        self._unload_diffusion()

        from leffa.inference import LeffaInference
        from leffa.model import LeffaModel

        model = LeffaModel(
            pretrained_model_name_or_path=str(
                self.ckpt_dir / "stable-diffusion-xl-1.0-inpainting-0.1"
            ),
            pretrained_model=str(self.ckpt_dir / "pose_transfer.pth"),
            dtype=self.dtype,
        )
        self._pt_inference = LeffaInference(model=model)
        self._active_model = "pose"
        print("Loaded pose-transfer model")

    def _get_face_app(self):
        if self._face_app is not None:
            return self._face_app
        try:
            from insightface.app import FaceAnalysis

            app = FaceAnalysis(
                name="buffalo_l",
                providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
            )
            app.prepare(ctx_id=0, det_size=(640, 640))
            self._face_app = app
        except Exception as exc:
            print(f"[face_lock] Could not init InsightFace: {exc}")
            self._face_app = False
        return self._face_app if self._face_app is not False else None

    # --------------------------------------------------------------- inference
    def _run_control(
        self,
        src_image: Image.Image,
        ref_image: Image.Image,
        control_type: Literal["virtual_tryon", "pose_transfer"],
        step: int = 30,
        scale: float = 2.5,
        seed: int = 42,
        ref_acceleration: bool = True,
        vt_model_type: str = "viton_hd",
        vt_garment_type: str = "upper_body",
        vt_repaint: bool = False,
    ) -> Image.Image:
        from leffa.transform import LeffaTransform
        from leffa_utils.utils import (
            get_agnostic_mask_dc,
            get_agnostic_mask_hd,
            resize_and_center,
        )

        self._load_preprocessors()

        src_image = resize_and_center(src_image.convert("RGB"), 768, 1024)
        ref_image = resize_and_center(ref_image.convert("RGB"), 768, 1024)
        src_array = np.array(src_image)

        if control_type == "virtual_tryon":
            model_parse, _ = self._parsing(src_image.resize((384, 512)))
            keypoints = self._openpose(src_image.resize((384, 512)))
            if vt_model_type == "viton_hd":
                mask = get_agnostic_mask_hd(model_parse, keypoints, vt_garment_type)
            else:
                mask = get_agnostic_mask_dc(model_parse, keypoints, vt_garment_type)
            mask = mask.resize((768, 1024))

            if vt_model_type == "viton_hd":
                seg = self._densepose.predict_seg(src_array)[:, :, ::-1]
            else:
                iuv = self._densepose.predict_iuv(src_array)
                seg = np.concatenate([iuv[:, :, 0:1]] * 3, axis=-1)
            densepose = Image.fromarray(seg)
            self._load_vton(vt_model_type)
            inference = self._vt_inference
        else:
            mask = Image.fromarray(np.ones_like(src_array) * 255)
            iuv = self._densepose.predict_iuv(src_array)[:, :, ::-1]
            densepose = Image.fromarray(iuv)
            self._load_pose()
            inference = self._pt_inference

        data = {
            "src_image": [src_image],
            "ref_image": [ref_image],
            "mask": [mask],
            "densepose": [densepose],
        }
        data = LeffaTransform()(data)
        output = inference(
            data,
            ref_acceleration=ref_acceleration,
            num_inference_steps=int(step),
            guidance_scale=float(scale),
            seed=int(seed),
            repaint=vt_repaint if control_type == "virtual_tryon" else False,
        )
        return output["generated_image"][0].convert("RGB")

    def generate(
        self,
        base_image: PathLike,
        ref_image: PathLike,
        mode: Mode = "both",
        garment_type: str = "upper_body",
        vt_model_type: str = "viton_hd",
        steps: int = 30,
        guidance_scale: float = 2.5,
        seed: int = 42,
        ref_acceleration: bool = True,
        face_lock: Optional[bool] = None,
        return_debug: bool = False,
    ) -> Image.Image | Tuple[Image.Image, dict]:
        """
        Run outfit and/or pose transfer.

        Leffa pose-transfer convention
        ------------------------------
        src = target pose image
        ref = appearance (identity / clothes) image
        """
        base = _as_pil(base_image)
        ref = _as_pil(ref_image)
        do_face = self.enable_face_lock if face_lock is None else face_lock
        debug: dict = {}

        current = base

        if mode in ("both", "outfit_only"):
            print("Step 1/2: Outfit transfer (VTON) — clothes from ref onto base...")
            # Person = base, garment/person-wearing-clothes = ref
            current = self._run_control(
                src_image=current,
                ref_image=ref,
                control_type="virtual_tryon",
                step=steps,
                scale=guidance_scale,
                seed=seed,
                ref_acceleration=ref_acceleration,
                vt_model_type=vt_model_type,
                vt_garment_type=garment_type,
            )
            debug["after_vton"] = current.copy()
            self._unload_diffusion()

        if mode in ("both", "pose_only"):
            print("Step 2/2: Pose transfer — appearance from dressed base, pose from ref...")
            # Appearance = current (base or dressed), pose = ref
            appearance = current if mode == "both" else base
            current = self._run_control(
                src_image=ref,
                ref_image=appearance,
                control_type="pose_transfer",
                step=steps,
                scale=guidance_scale,
                seed=seed,
                ref_acceleration=ref_acceleration,
            )
            debug["after_pose"] = current.copy()
            self._unload_diffusion()

        if do_face and mode != "outfit_only":
            # After pose warp, re-apply base face. For outfit-only, Leffa usually
            # already keeps the face; still allow explicit lock if requested.
            print("Face identity lock...")
            current = lock_face_identity(
                base_image=base,
                generated_image=current,
                face_app=self._get_face_app(),
            )
            debug["after_face_lock"] = current.copy()
        elif do_face and mode == "outfit_only":
            print("Face identity lock (outfit mode)...")
            current = lock_face_identity(
                base_image=base,
                generated_image=current,
                face_app=self._get_face_app(),
                blend=0.75,
            )
            debug["after_face_lock"] = current.copy()

        free_vram()
        if return_debug:
            return current, debug
        return current

    def generate_from_paths(
        self,
        base_path: str,
        ref_path: str,
        **kwargs,
    ) -> Image.Image:
        return self.generate(base_path, ref_path, **kwargs)  # type: ignore[return-value]


In [ ]:
%%writefile /content/pose_cloth_swap/pipeline/__init__.py
"""Pose & Cloth Swap pipeline helpers for Colab / local use."""

from .face_lock import lock_face_identity
from .memory import free_vram
from .leffa_sequential import PoseClothPipeline

__all__ = [
    "PoseClothPipeline",
    "lock_face_identity",
    "free_vram",
]


In [ ]:
!ls -l /content/pose_cloth_swap/pipeline/


## 4. Download model weights

First run downloads several GB from Hugging Face. Re-run only if the session reset and files were wiped.


In [ ]:
import sys
from pathlib import Path

WORK = Path("/content/pose_cloth_swap")
sys.path.insert(0, str(WORK))
sys.path.insert(0, str(WORK / "Leffa"))

# Ensure detectron2 path survives if using clone fallback
_d2 = Path("/content/pose_cloth_swap/detectron2_path.txt")
if _d2.exists():
    _p = _d2.read_text().strip()
    if _p and _p not in sys.path:
        sys.path.insert(0, _p)

from pipeline import PoseClothPipeline

pipe = PoseClothPipeline(
    leffa_root=str(WORK / "Leffa"),
    ckpt_dir=str(WORK / "Leffa" / "ckpts"),
    dtype="float16",
    enable_face_lock=True,
)
pipe.download_checkpoints()
print("Pipeline object ready.")


## 5. Launch Gradio app

- **Base image** = person to keep (face + body proportions)
- **Pose & Outfit image** = pose + clothes to copy
- Mode **Both** runs outfit → pose → face lock


In [ ]:
import gradio as gr
from PIL import Image, ImageDraw
from pathlib import Path
import sys

WORK = Path("/content/pose_cloth_swap")
sys.path.insert(0, str(WORK))
sys.path.insert(0, str(WORK / "Leffa"))

# Ensure detectron2 path survives if using clone fallback
_d2 = Path("/content/pose_cloth_swap/detectron2_path.txt")
if _d2.exists():
    _p = _d2.read_text().strip()
    if _p and _p not in sys.path:
        sys.path.insert(0, _p)

try:
    pipe
except NameError:
    from pipeline import PoseClothPipeline
    pipe = PoseClothPipeline(
        leffa_root=str(WORK / "Leffa"),
        ckpt_dir=str(WORK / "Leffa" / "ckpts"),
        dtype="float16",
        enable_face_lock=True,
    )

OUT_DIR = WORK / "outputs"
OUT_DIR.mkdir(exist_ok=True)


def _side_by_side(base, ref, result):
    imgs = [im.convert("RGB").resize((384, 512)) for im in (base, ref, result)]
    canvas = Image.new("RGB", (384 * 3 + 20, 512 + 36), (245, 245, 245))
    labels = ["Base", "Pose & Outfit", "Result"]
    draw = ImageDraw.Draw(canvas)
    for i, (im, lab) in enumerate(zip(imgs, labels)):
        x = i * 384 + i * 10
        canvas.paste(im, (x, 28))
        draw.text((x + 8, 6), lab, fill=(20, 20, 20))
    return canvas


def run_swap(base, ref, mode, garment_type, steps, guidance, seed, face_lock, progress=gr.Progress(track_tqdm=True)):
    if base is None or ref is None:
        raise gr.Error("Please upload both a Base image and a Pose & Outfit image.")
    mode_map = {
        "Both (outfit + pose)": "both",
        "Outfit only": "outfit_only",
        "Pose only": "pose_only",
    }
    garment_map = {
        "Upper body": "upper_body",
        "Lower body": "lower_body",
        "Dress / full": "dresses",
    }
    progress(0.05, desc="Starting pipeline...")
    result = pipe.generate(
        base_image=base,
        ref_image=ref,
        mode=mode_map[mode],
        garment_type=garment_map[garment_type],
        steps=int(steps),
        guidance_scale=float(guidance),
        seed=int(seed),
        ref_acceleration=True,
        face_lock=bool(face_lock),
    )
    progress(0.95, desc="Saving...")
    out_path = OUT_DIR / "result.png"
    result.save(out_path)
    compare = _side_by_side(base, ref, result)
    compare.save(OUT_DIR / "compare.png")
    return result, compare, str(out_path)


with gr.Blocks(title="Pose & Cloth Swap") as demo:
    gr.Markdown(
        """
        # Pose & Cloth Swap
        Keep the **base** person's face and body proportions. Copy **pose + outfit** from the reference image.

        Free Colab build · Leffa (VTON + pose) · InsightFace face lock
        """
    )
    with gr.Row():
        base_in = gr.Image(label="Base image (identity to keep)", type="pil", height=360)
        ref_in = gr.Image(label="Pose & Outfit image", type="pil", height=360)
    with gr.Row():
        mode = gr.Radio(
            ["Both (outfit + pose)", "Outfit only", "Pose only"],
            value="Both (outfit + pose)",
            label="Mode",
        )
        garment = gr.Radio(
            ["Upper body", "Lower body", "Dress / full"],
            value="Upper body",
            label="Garment region (for outfit transfer)",
        )
    with gr.Accordion("Advanced", open=False):
        steps = gr.Slider(20, 50, value=30, step=1, label="Inference steps")
        guidance = gr.Slider(1.0, 5.0, value=2.5, step=0.1, label="Guidance scale")
        seed = gr.Number(value=42, precision=0, label="Seed")
        face_lock = gr.Checkbox(value=True, label="Face identity lock")
    btn = gr.Button("Generate", variant="primary")
    with gr.Row():
        result_out = gr.Image(label="Result", type="pil", height=420)
        compare_out = gr.Image(label="Base | Ref | Result", type="pil", height=420)
    file_out = gr.File(label="Download result PNG")

    btn.click(
        fn=run_swap,
        inputs=[base_in, ref_in, mode, garment, steps, guidance, seed, face_lock],
        outputs=[result_out, compare_out, file_out],
    )

demo.queue(max_size=2).launch(share=True, debug=True)


## Tips

| Issue | Fix |
|---|---|
| CUDA OOM | Use **Outfit only** or **Pose only**, or lower steps to 20 |
| Accidental restart prompt | You imported packages before install — **Disconnect and delete runtime**, reopen, run from section 1 |
| Session disconnected | Re-run from section 1 |
| Face looks off | Clear frontal face on **base**; keep Face lock on |
| Clothes wrong region | Set Garment region to Upper / Lower / Dress |

**Note:** Diffusion cannot guarantee literal 100% identity. Face lock maximizes consistency for free Colab use.
